# Notebook 12: Edge Analysis and Fee-Adjusted Profitability

Identifies games where model probability diverges from Kalshi opening price by a meaningful margin, and computes fee-adjusted profitability for simulated $1/contract flat bets.

**Standalone:** This notebook loads `predictions_2025.parquet` from disk -- does not re-run training.

**Fee model:** 7% of profits on winning trades (`KALSHI_FEE_RATE = 0.07`). No fee on losses. Flat $1/contract sizing.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.edge import KALSHI_FEE_RATE, compute_edge_signals, compute_fee_adjusted_pnl
from src.data.kalshi import fetch_kalshi_markets, fetch_kalshi_open_prices

## Step 1: Load Predictions and Kalshi Data

Loads `predictions_2025.parquet` (from notebook 11) and joins with Kalshi opening prices.

In [ ]:
# Standalone load -- no training code imported
predictions = pd.read_parquet("../data/results/predictions_2025.parquet")
print(f"Loaded {len(predictions)} prediction rows ({len(predictions) // 3} games x 3 models)")

# Load Kalshi data with open prices
kalshi_df = fetch_kalshi_markets(max_age_hours=float("inf"))
kalshi_df = fetch_kalshi_open_prices(kalshi_df)

# Date alignment
kalshi_df["game_date"] = pd.to_datetime(kalshi_df["date"])

# Track fallback status BEFORE filling NaN
kalshi_df["is_fallback_price"] = kalshi_df["kalshi_open_price"].isna()
n_fallback = kalshi_df["is_fallback_price"].sum()

# Fill NaN open prices with closing price
kalshi_df["kalshi_open_price"] = kalshi_df["kalshi_open_price"].fillna(kalshi_df["kalshi_yes_price"])

# Drop doubleheader duplicates
kalshi_deduped = kalshi_df.drop_duplicates(subset=["game_date", "home_team", "away_team"], keep="first")

# Merge
merged = predictions.merge(
    kalshi_deduped[["game_date", "home_team", "away_team", "kalshi_open_price", "is_fallback_price"]],
    on=["game_date", "home_team", "away_team"],
    how="inner",
)
print(f"Matched: {len(merged)} rows")
print(f"Fallback price games: {n_fallback}")

# Caveat check
if n_fallback > 0:
    print()
    print("WARNING: Some games use closing price (fallback). For those games:")
    print('  \"closing price benchmark is invalid for edge analysis; Brier score comparison only\"')

## Step 2: Edge Identification

Configurable threshold (default 5 percentage points). An "edge" exists when `|model_prob - kalshi_open_price| > threshold`.

**Adjust the threshold below to explore different sensitivity levels:**

In [ ]:
# === CONFIGURABLE PARAMETERS ===
edge_threshold = 0.05  # Change this to explore: 0.03, 0.05, 0.10
# ===============================

print(f"Edge threshold: {edge_threshold:.0%} ({edge_threshold*100:.0f} percentage points)")
print(f"Fee rate: {KALSHI_FEE_RATE:.0%}")
print()

# Compute edges per model -- exclude fallback-price rows from edge analysis
valid_mask = ~merged["is_fallback_price"]
valid_merged = merged[valid_mask].copy()
fallback_merged = merged[~valid_mask].copy()

if len(fallback_merged) > 0:
    print(f"Excluding {len(fallback_merged)} fallback-price rows from edge analysis.")
    print('  (Closing price benchmark is invalid for edge analysis; Brier score comparison only.)')
    print()

edge_summary = []
for model_name in ["lr", "rf", "xgb"]:
    model_df = valid_merged[valid_merged["model_name"] == model_name].copy()
    model_edges = compute_edge_signals(model_df, threshold=edge_threshold)
    flagged = model_edges[model_edges["has_edge"]]
    edge_summary.append({
        "model": model_name,
        "total_games": len(model_df),
        "edge_games": len(flagged),
        "pct_edges": f"{100 * len(flagged) / len(model_df):.1f}%" if len(model_df) > 0 else "0.0%",
        "avg_abs_edge": f"{flagged['abs_edge'].mean():.3f}" if len(flagged) > 0 else "N/A",
        "buy_yes": (flagged["position"] == "BUY_YES").sum() if len(flagged) > 0 else 0,
        "buy_no": (flagged["position"] == "BUY_NO").sum() if len(flagged) > 0 else 0,
    })

print("=== Edge Summary by Model ===")
print(pd.DataFrame(edge_summary).to_string(index=False))

## Step 3: Fee-Adjusted Profitability

For each edge game, simulates a $1 flat bet in the direction of the edge signal and computes net P&L after Kalshi's 7% fee on profits.

**Position logic:**
- `BUY_YES` when model > market (model thinks home team more likely to win)
- `BUY_NO` when model < market (model thinks home team less likely to win)

In [ ]:
pnl_summary = []
for model_name in ["lr", "rf", "xgb"]:
    model_df = valid_merged[valid_merged["model_name"] == model_name].copy()
    model_edges = compute_edge_signals(model_df, threshold=edge_threshold)
    flagged = model_edges[model_edges["has_edge"]].copy()

    if len(flagged) == 0:
        pnl_summary.append({
            "model": model_name,
            "n_bets": 0,
            "gross_pnl": 0.0,
            "net_pnl": 0.0,
            "win_rate": "N/A",
            "roi": "N/A",
        })
        continue

    # Compute P&L for each edge bet
    flagged["pnl"] = flagged.apply(compute_fee_adjusted_pnl, axis=1)
    wins = flagged["pnl"] > 0
    total_wagered = len(flagged)  # $1 per bet

    pnl_summary.append({
        "model": model_name,
        "n_bets": len(flagged),
        "wins": wins.sum(),
        "losses": (~wins).sum(),
        "win_rate": f"{100 * wins.mean():.1f}%",
        "total_pnl": f"${flagged['pnl'].sum():.2f}",
        "avg_pnl_per_bet": f"${flagged['pnl'].mean():.4f}",
        "roi": f"{100 * flagged['pnl'].sum() / total_wagered:.1f}%",
    })

print(f"=== Fee-Adjusted Profitability (threshold={edge_threshold:.0%}, fee={KALSHI_FEE_RATE:.0%}) ===")
print(f"Position sizing: $1/contract flat bet on every edge signal")
print()
print(pd.DataFrame(pnl_summary).to_string(index=False))

## Step 4: Edge Distribution Visualization

Histogram of edge magnitudes (model_prob - kalshi_open_price) across all models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, model_name in zip(axes, ["lr", "rf", "xgb"]):
    model_df = valid_merged[valid_merged["model_name"] == model_name].copy()
    model_edges = compute_edge_signals(model_df, threshold=edge_threshold)

    edges = model_edges["edge"]
    flagged = model_edges[model_edges["has_edge"]]["edge"]

    ax.hist(edges, bins=40, alpha=0.6, color="steelblue", label="All games")
    if len(flagged) > 0:
        ax.hist(flagged, bins=20, alpha=0.8, color="tomato", label=f"Edge (>{edge_threshold:.0%})")
    ax.axvline(x=0, color="black", linewidth=0.5)
    ax.axvline(x=edge_threshold, color="red", linewidth=0.5, linestyle="--")
    ax.axvline(x=-edge_threshold, color="red", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Edge (model - market)")
    ax.set_title(f"{model_name.upper()}")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Count")
fig.suptitle(f"Edge Distribution (threshold={edge_threshold:.0%})", fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Cumulative P&L Over Time

Shows how each model's edge-betting strategy performs game-by-game through the season.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for model_name in ["lr", "rf", "xgb"]:
    model_df = valid_merged[valid_merged["model_name"] == model_name].copy()
    model_edges = compute_edge_signals(model_df, threshold=edge_threshold)
    flagged = model_edges[model_edges["has_edge"]].copy()

    if len(flagged) == 0:
        continue

    flagged["pnl"] = flagged.apply(compute_fee_adjusted_pnl, axis=1)
    flagged = flagged.sort_values("game_date")
    flagged["cumulative_pnl"] = flagged["pnl"].cumsum()
    ax.plot(flagged["game_date"], flagged["cumulative_pnl"], label=model_name.upper())

ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xlabel("Game Date")
ax.set_ylabel("Cumulative P&L ($)")
ax.set_title(f"Cumulative Fee-Adjusted P&L (threshold={edge_threshold:.0%}, fee={KALSHI_FEE_RATE:.0%}, $1/bet)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

- **Edge threshold:** Configurable (default 5pp). Re-run cell 6 with different values to explore.
- **Fee model:** `KALSHI_FEE_RATE = 0.07` (7% of profits). Named constant in `src/models/edge.py`.
- **Position sizing:** Flat $1 per contract. No Kelly sizing.
- **Caveats:** Games using fallback closing price are excluded from edge analysis. Closing price reflects the outcome and is invalid for edge detection.

> **Note:** The simplified fee formula (`0.07 * profit`) approximates Kalshi's actual quadratic fee schedule. The constant is easy to update in `src/models/edge.py` when the actual MLB-specific fee rate is verified.